<a href="https://colab.research.google.com/github/tomisagoodguy/tomisagoodguy.github.io/blob/main/%E5%AE%9A%E5%BA%B7%E6%B5%81%E4%BA%94%E6%98%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## pip

In [ ]:
%%capture
!pip install finlab>log.txt
!pip install ta-lib-bin>log.txt
#!pip install git+https://github.com/microsoft/qlib.git>log.txt

In [ ]:
import finlab
#line_access_token = ["hxlnkinXYMD5gGHOTOSqfBhSULSLNcW6Z6Sb7z6Gf4R"]
from google.colab import userdata
finlab.login(userdata.get('FINLAB_API_TOKEN'))
# 儲存csv檔
#report.get_trades().to_csv('/content/drive/MyDrive/黑皮.csv')
#open_close_avg

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
#import plotly.express as px
from scipy.stats import rankdata
from finlab.dataframe import FinlabDataFrame
from finlab import dataframe
from finlab import login
import os
from finlab import backtest
from finlab.online.order_executor import Position
import matplotlib.pyplot as plt
data.set_storage(data.CacheStorage())


# small cap 0050

In [ ]:
# plt.style.reload_library()

plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pitayasmoothie-dark.mplstyle')
# plt.style.use('https://github.com/kimichenn/nord-deep-mpl-stylesheet/raw/main/nord-deep.mplstyle')


price = data.get('etl:adj_close')
s1 = price.pct_change().mean(axis=1)
s2 = price.pct_change()['0050']

val = ((s1 - s2).rolling(240).mean().add(1) ** 240) - 1

# line thin
plt.rcParams['lines.linewidth'] = 2
val.plot()

plt.legend(['Small Cap Alpha to 0050 (Rolling Year)', ''])

# market the current value
plt.axhline(val.iloc[-1], color='yellow', linestyle='--')
plt.axhline(0, color='white', linestyle='--')

# mark the circle
plt.plot(val.index[-1], val.iloc[-1], 'o', color='yellow')

# show the current value and bold text
plt.text(val.index[-1], val.iloc[-1]-0.07, round(val.iloc[-1], 2), color='yellow', fontsize=16, fontweight='bold')


# use percentage as y
import matplotlib.ticker as mtick
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# change font to arial
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['font.weight'] = 'bold'

# change dpi
plt.rcParams['figure.dpi'] = 200


In [ ]:
#price['0050'].plot()

# 投信

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame

close = data.get('price:收盤價')
投信買賣超股數 = data.get('institutional_investors_trading_summary:投信買賣超股數')
投信日累計買賣超股數 = 投信買賣超股數.rolling(15).sum()
普通股股本 = data.get('financial_statement:普通股股本')
volume = data.get("price:成交股數")
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())



position = (
        ((投信日累計買賣超股數*close.average(15)).is_largest(250)) &
        ((rev_ma2/rev_ma2.shift(12)).rank(pct=True, axis=1)>=0.6)&
        (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.6)&
        (close > close.average(5)) &
        (close.average(6) > close.average(21))&
        ((turnover_rate.average(5).rank(pct=True, axis=1) > 0.6).sustain(3,2)) &
        ((data.get('margin_transactions:融資今日餘額').average(5).rank(pct=True) < 0.3).sustain(5,3))

            )


position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='close',
        name='投信買超_test',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
#report.display_mae_mfe_analysis()

# 投信市值小

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame

close = data.get('price:收盤價')
投信買賣超股數 = data.get('institutional_investors_trading_summary:投信買賣超股數')
投信日累計買賣超股數 = 投信買賣超股數.rolling(15).sum()
普通股股本 = data.get('financial_statement:普通股股本')
volume = data.get("price:成交股數")
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
cap = data.get('etl:market_value')

rev_ma2=rev.average(2)
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
mrev_2m_yoyg = (rev_ma2 - rev_ma2.shift(12)) / abs(rev_ma2.shift(12).where(rev_ma2.shift(12)!=0)) * 100
mrev_2m_yoyg_per_price = mrev_2m_yoyg / data.get('price:收盤價')
roe_per_price = data.get('fundamental_features:ROE稅後') / data.get('price:收盤價')
rsv = (close - close.rolling(60).min()) / (close.rolling(60).max() - close.rolling(60).min())
#當日沖銷交易成交股數 = data.get("intraday_trading:當日沖銷交易成交股數")


position =close[
        ((投信日累計買賣超股數*close.average(15)).is_largest(250)) &
        (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.6)&
        (mrev_2m_yoyg.rank(pct=True, axis=1)>=0.6)&
        #mrev_2m_yoyg_per_price.is_largest(400) &
        (close > close.average(5)) &
        (close.average(6) > close.average(21))&
        ((turnover_rate.average(5).rank(pct=True, axis=1) > 0.6).sustain(3,2)) &
        (rsv.rank(pct=True, axis=1) > 0.65) &
       # (((當日沖銷交易成交股數/volume/2)< 0.2).sustain(5,2)) &
        ((data.get('margin_transactions:融資今日餘額').average(5).rank(pct=True) < 0.3).sustain(5,4))].is_smallest(5)


position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='open_close_avg',
        name='投信買超_testV2',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
report.display_mae_mfe_analysis()

# 投信殖利率

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame

close = data.get('price:收盤價')
投信買賣超股數 = data.get('institutional_investors_trading_summary:投信買賣超股數')
投信日累計買賣超股數 = 投信買賣超股數.rolling(10).sum()
普通股股本 = data.get('financial_statement:普通股股本')
volume = data.get("price:成交股數")
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rsv = (close - close.rolling(60).min()) / (close.rolling(60).max() - close.rolling(60).min())
ope_earn = data.get('fundamental_features:營業利益率')
ytd = data.get('price_earning_ratio:殖利率(%)')


position =close[
        ((投信日累計買賣超股數*close.average(10)).is_largest(300)) &
        (data.get('price_earning_ratio:殖利率(%)')>=4)&
        (ope_earn >= ope_earn.quantile_row(0.25))&
        (close > close.average(5)) &
        (close.average(6) > close.average(21))&
        ((turnover_rate.average(10).rank(pct=True, axis=1) > 0.6).sustain(3,2)) &
        (rsv.rank(pct=True, axis=1) > 0.7) &
        ((data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.3).sustain(5,3))
        ].is_smallest(5)


position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='open_close_avg',
        name='投信買超_testV3',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
#report.display_mae_mfe_analysis()

# perprice

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest


close = data.get("price:收盤價")
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
mrev_2m_yoyg = (rev_ma2 - rev_ma2.shift(12)) / abs(rev_ma2.shift(12).where(rev_ma2.shift(12)!=0)) * 100
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())
roe = data.get('fundamental_features:經常稅後淨利')/data.get('financial_statement:股東權益總額')

# 買入條件
position = (
        (roe.rank(pct=True, axis=1)>=0.6) &
        ((rev_ma2/rev_ma2.shift(12)).rank(pct=True, axis=1)>=0.6) &
        (data.get('price:成交股數').average(5) > 500*1000)

)

# 整合部位
position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


# 回測

report = sim(position,
        trade_at_price='close',
        name='perpriceanalysis',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)

#report.display_mae_mfe_analysis()

# 外資

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame

close = data.get('price:收盤價')
外資 = data.get('institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)')
外資日累計買賣超股數 = 外資.rolling(20).sum()
普通股股本 = data.get('financial_statement:普通股股本')
volume = data.get("price:成交股數")
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())

position = (
        ((外資日累計買賣超股數*close.average(20)).is_largest(300)) &
        ((rev_ma2/rev_ma2.shift(12)).rank(pct=True, axis=1)>=0.6)&
        (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.6)&
        (close > close.average(5)) &
        (close.average(6) > close.average(21))&
        ((turnover_rate.average(5).rank(pct=True, axis=1) > 0.6).sustain(3,2)) &
        ((data.get('margin_transactions:融資今日餘額').average(5).rank(pct=True) < 0.25).sustain(5,3))

            )


position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='open_close_avg',
        name='外資買超_test',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
#report.display_mae_mfe_analysis()

# 外資市值小

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame

close = data.get('price:收盤價')
外資 = data.get('institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)')
外資日累計買賣超股數 = 外資.rolling(20).sum()
普通股股本 = data.get('financial_statement:普通股股本')
volume = data.get("price:成交股數")
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())
cap = data.get('etl:market_value')


position = cap[
        ((外資日累計買賣超股數*close.average(20)).is_largest(300)) &
        ((rev_ma2/rev_ma2.shift(12)).rank(pct=True, axis=1)>=0.6)&
        (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.6)&
        (close > close.average(5)) &
        (close.average(6) > close.average(21))&
        (rsv.rank(pct=True, axis=1) > 0.65) &
        ((turnover_rate.average(5).rank(pct=True, axis=1) > 0.6).sustain(3,2)) &
        ((data.get('margin_transactions:融資今日餘額').average(5).rank(pct=True) < 0.25).sustain(5,3))

].is_smallest(5)


#position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='close',
        name='外資買超_testV2',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
report.display_mae_mfe_analysis()

## #黑暗騎士2

In [ ]:
from finlab import data, backtest
from finlab.dataframe import FinlabDataFrame
import numpy as np

# 获取必要数据
cap = data.get('etl:market_value')
vol = data.get('price:成交股數')
普通股股本 = data.get('financial_statement:普通股股本')
turnover_rate=(vol) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
mrev_2m_yoyg = (rev_ma2 - rev_ma2.shift(12)) / abs(rev_ma2.shift(12).where(rev_ma2.shift(12)!=0)) * 100

rev_year_growth = data.get('monthly_revenue:去年同月增減(%)')
close = data.get('price:收盤價')
inv = data.get('inventory')
df1 = data.get('financial_statement:投資活動之淨現金流入_流出')
df2 = data.get('financial_statement:營業活動之淨現金流入_流出')
稅後淨利 = data.get('fundamental_features:經常稅後淨利')
權益總計 = data.get('financial_statement:股東權益總額')
稅後淨利成長率 = data.get('fundamental_features:稅後淨利成長率')
high = data.get("price:最高價")
自由現金流 = (df1 + df2).rolling(4).mean()
股東權益報酬率 = 稅後淨利 / 權益總計
rev = data.get('monthly_revenue:當月營收') * 1000
當季營收 = rev.rolling(4).sum()
市值營收比 = cap / 當季營收
rev_ma2=rev.average(2)
rev_ma3 = rev.average(3)
rev_ma11 = rev.average(11)

# 计算持股分级数据
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
                     .reset_index().groupby(['date', 'stock_id'])
                     .agg({'持有股數': 'sum'})
                     .reset_index()
                     .pivot(index='date', columns='stock_id', values='持有股數'))

h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) & (inv.持股分級.astype(int) <= 14)]
                     .reset_index().groupby(['date', 'stock_id'])
                     .agg({'持有股數': 'sum'})
                     .reset_index()
                     .pivot(index='date', columns='stock_id', values='持有股數'))

ratio = (h2 / (h1 + h2))

# 计算选股条件的评分
s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) * (close.notna()))

# 计算市场基准
with data.universe(market="OTC"):
    otc_close = data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)

RS = ((close / close.shift(3)) / (benchmark / benchmark.shift(3)))

new_low = close == close.rolling(20).min()
no_new_low_days = new_low.rolling(4).sum() == 0

# 策略1: 构建选股条件
position = (
    (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.55)&
    (mrev_2m_yoyg.rank(pct=True, axis=1)>=0.55)&
    (cap < cap.quantile_row(0.8)) &
    (close > close.average(5)) &
    (close > close.average(240)) &
    (close.average(6) > close.average(21)) &
    (turnover_rate.average(5) > 0.8) &
    (RS.rank(pct=True, axis=1) > 0.6) &
    (s.rank(pct=True, axis=1) > 0.25) &
   # ~(rev_year_growth < 10).sustain(3) &
    no_new_low_days &
    (data.indicator("RSI", timeperiod=10) < 90) &
    (data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.1).sustain(6, 4)
)

# 计算RSV指标
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())

# 根据RSV选择前5大持仓
position = (position[position > 0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)

# 策略2: 计算技术指标
def compute_candle_volatility(timeperiod=20):
    close = data.get("price:收盤價")
    high = data.get("price:最高價")
    low = data.get("price:最低價")
    open_ = data.get("price:開盤價")

    bullish_candle = close >= open_
    bullish_volatility = abs(close.shift() - open_) + abs(open_ - low) + abs(low - high) + abs(high - close)
    bearish_volatility = abs(close.shift() - open_) + abs(open_ - high) + abs(high - low) + abs(low - close)
    candle_volatility = FinlabDataFrame(np.nan, index=close.index, columns=close.columns)
    candle_volatility[bullish_candle] = bullish_volatility
    candle_volatility[~bullish_candle] = bearish_volatility
    volatility = candle_volatility.average(timeperiod) / close.average(timeperiod) * 100
    return volatility

volatility = compute_candle_volatility()

# 计算持股分级数据
h4 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) & (inv.持股分級.astype(int) <= 15)]
                     .reset_index().groupby(['date', 'stock_id'])
                     .agg({'持有股數': 'sum'})
                     .reset_index()
                     .pivot(index='date', columns='stock_id', values='持有股數'))

ratio1 = (h4 / (h1 + h4))

# 计算选股条件的评分
s1 = (FinlabDataFrame(ratio1.diff(6)).rank(axis=1, pct=True) * (close.notna()))

# 计算其他技术指标
new_low = close == close.rolling(20).min()
no_new_low_2days = new_low.rolling(2).sum() == 0
short_volatility = compute_candle_volatility(timeperiod=10)
long_volatility = compute_candle_volatility(timeperiod=60)

# 策略2: 构建选股条件
position2 = (
    (自由現金流 > 0) &
    (股東權益報酬率 > 0) &
    (稅後淨利成長率 > 0) &
    (市值營收比 < 3) &
    (cap < cap.quantile_row(0.75)) &
    (cap > cap.quantile_row(0.1)) &
    (s1.rank(pct=True, axis=1) > 0.3) &
    (rev / rev.shift(12) > 1.05) &
    (rev_ma3 / rev_ma11 > 0.95) &
    (volatility <= 7.5) &
    (vol > 150 * 1000) &
    (close.average(5) > close.average(90)) &
    no_new_low_2days &
    (RS.rank(pct=True, axis=1) > 0.5)
)

# 筛选卖出条件
high_rolling1 = high.rolling(10).max()
high_rolling2 = high.rolling(20).max()
sell_cond1 = close < high_rolling1 * 0.9
sell_cond2 = (close < high_rolling2 - 3 * long_volatility) & (short_volatility / close.average(10) * 100 > 6)
sell_cond3 = close.rolling(10).mean() < close.rolling(20).mean() * 0.9
sell_condition = sell_cond1 | sell_cond2 | sell_cond3
position2[sell_condition] = False
position2 = (position2[position2 > 0] * rsv).is_largest(5)
position2 = position2.reindex(rev.index_str_to_date().index)

# 综合策略1和策略2的持仓
new_position = (position * 0.5 + position2 * 0.5)

# 回测策略
report = backtest.sim(
    new_position,
    stop_loss=0.11,
    position_limit=0.3,
    trade_at_price='close',
    upload=True,
    name='黑暗騎士2',
    fee_ratio=1.425 / 1000 * 0.22,
    mae_mfe_window=40,
    live_performance_start='2023-09-01'
)


In [ ]:
#report.display_mae_mfe_analysis()

# 抬轎一

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest




市值 = data.get('etl:market_value')
volume = data.get("price:成交股數")
普通股股本 = data.get('financial_statement:普通股股本')
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
rev_ma2=rev.average(2)
#rev_ma3 = rev.rolling(3).mean()
#rev_ma12 = rev.rolling(12).mean()
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
mrev_2m_yoyg = (rev_ma2 - rev_ma2.shift(12)) / abs(rev_ma2.shift(12).where(rev_ma2.shift(12)!=0)) * 100
#mrev_3m_yoyg = (rev_ma3 - rev_ma3.shift(12)) / abs(rev_ma3.shift(12).where(rev_ma3.shift(12)!=0)) * 100

close = data.get('price:收盤價')
融資今日餘額 = data.get("margin_transactions:融資今日餘額")

inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 14)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))


ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
    (close.notna()))


with data.universe(market="OTC"):
    otc_close = data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)

#RS = ((close/close.shift(5)) / (benchmark/benchmark.shift(5)))
RS1 = ((close/close.shift(3)) / (benchmark/benchmark.shift(3)))

rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())
new_low = close == close.rolling(20).min()
no_new_low_days = new_low.rolling(4).sum() == 0


position= (

            (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.55) &
            (mrev_2m_yoyg.rank(pct=True, axis=1)>=0.55)&
            (市值 < 市值.quantile_row(0.8))&
            (close > close.average(5)) &
            (close > close.average(240)) &
            (close.average(6) > close.average(21))&
            (no_new_low_days) &
            (turnover_rate.average(10) > 0.8) &
            ((RS1.rank(pct=True, axis=1) > 0.6))&
            ((s.rank(pct=True, axis=1) > 0.25))&
           #有成交股數門檻比較符合現實
            (data.get('price:成交股數').average(5) > 100*1000) &
            (data.indicator("RSI", timeperiod=10) < 90) &
            ((data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.1).sustain(6,4))
            )





position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


report = sim(
            position
            ,stop_loss=0.1
            ,take_profit=0.4
            ,name='抬轎一'
            ,trade_at_price='close'
            ,fee_ratio=1.425/1000*0.22
            ,upload=True
            ,mae_mfe_window=40
            ,live_performance_start='2024-05-01',
            )

import matplotlib.pyplot as plt
from finlab import data, backtest

# Set the matplotlib style
plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pitayasmoothie-dark.mplstyle')
#plt.style.use('https://github.com/kimichenn/nord-deep-mpl-stylesheet/raw/main/nord-deep.mplstyle')

# Get financial data
close = data.get('price:收盤價')
vol = data.get('price:成交股數')

# Define Bollinger Bands calculations
中軌 = close.rolling(window=40).mean()
標準差 = close.rolling(window=40).std()
上軌 = 中軌 + 1.2 * 標準差
下軌 = 中軌 - 1.2 * 標準差

# Get stock IDs from the position DataFrame (where there are holdings)
stock_ids = position.columns[position.iloc[-1]]

# Determine the number of subplots
num_plots = len(stock_ids)
num_cols = min(num_plots, 4)  # Limit to 4 columns
num_rows = (num_plots + num_cols - 1) // num_cols  # Calculate the necessary number of rows

# Create a figure and a set of subplots
fig, axs = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6*num_cols, 6*num_rows))
axs = axs.flatten()  # Flatten the 2D array of axes for easy iteration

# Plot Bollinger Bands and Close Price for all stocks
for i, stock_id in enumerate(stock_ids):
    ax = axs[i]

    ax.plot(上軌[stock_id].loc['2024'], label='Upper Band')
    ax.plot(中軌[stock_id].loc['2024'], label='Middle Band')
    ax.plot(下軌[stock_id].loc['2024'], label='Lower Band')
    ax.plot(close[stock_id].loc['2024'], label='Close Price')

    ax.set_title(f'Bollinger Bands and Close Price for {stock_id}')
    ax.legend(loc='upper left')

    # Plot volume on the same axis but with a secondary y-axis
    ax2 = ax.twinx()
    ax2.bar(vol[stock_id].loc['2024'].index, vol[stock_id].loc['2024'], color='gray', alpha=0.3)
    ax2.set_ylabel('Volume')

# Hide any unused subplots
for j in range(i+1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()
report.display_mae_mfe_analysis()

# K線圖

In [ ]:
import matplotlib.pyplot as plt
from finlab import data, backtest

# Set the matplotlib style
plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pitayasmoothie-dark.mplstyle')
#plt.style.use('https://github.com/kimichenn/nord-deep-mpl-stylesheet/raw/main/nord-deep.mplstyle')

# Get financial data
close = data.get('price:收盤價')
vol = data.get('price:成交股數')

# Define Bollinger Bands calculations
中軌 = close.rolling(window=40).mean()
標準差 = close.rolling(window=40).std()
上軌 = 中軌 + 1.2 * 標準差
下軌 = 中軌 - 1.2 * 標準差

# Get stock IDs from the position DataFrame (where there are holdings)
stock_ids = position.columns[position.iloc[-1]]

# Determine the number of subplots
num_plots = len(stock_ids)
num_cols = min(num_plots, 4)  # Limit to 4 columns
num_rows = (num_plots + num_cols - 1) // num_cols  # Calculate the necessary number of rows

# Create a figure and a set of subplots
fig, axs = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6*num_cols, 6*num_rows))
axs = axs.flatten()  # Flatten the 2D array of axes for easy iteration

# Plot Bollinger Bands and Close Price for all stocks
for i, stock_id in enumerate(stock_ids):
    ax = axs[i]

    ax.plot(上軌[stock_id].loc['2024'], label='Upper Band')
    ax.plot(中軌[stock_id].loc['2024'], label='Middle Band')
    ax.plot(下軌[stock_id].loc['2024'], label='Lower Band')
    ax.plot(close[stock_id].loc['2024'], label='Close Price')

    ax.set_title(f'Bollinger Bands and Close Price for {stock_id}')
    ax.legend(loc='upper left')

    # Plot volume on the same axis but with a secondary y-axis
    ax2 = ax.twinx()
    ax2.bar(vol[stock_id].loc['2024'].index, vol[stock_id].loc['2024'], color='gray', alpha=0.3)
    ax2.set_ylabel('Volume')

# Hide any unused subplots
for j in range(i+1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()


# 分析

In [ ]:
import pandas as pd

df = report.get_trades()
years = [str(int(year)) for year in df['entry_date'].dt.year.dropna().unique()]
總獲利list = []
總虧損list = []
盈虧比list = []
平均每筆獲利list = []
平均每筆虧損list = []
平均每筆盈虧比list = []
總交易筆數list = []
獲利筆數list = []
獲利介於0至5的筆數佔總獲利筆數list = []
獲利大於10的筆數佔總獲利筆數list = []
獲利大於30的筆數佔總獲利筆數list = []
虧損筆數list = []
勝率list = []
yearlist = []

for y in years:
    cond1 = (df['return'] > 0) & (df['entry_date'].dt.year == int(y)) #獲利
    cond2 = (df['return'] < 0) & (df['entry_date'].dt.year == int(y)) #虧損

    cond3 = (df['return'] >= 0) & (df['return'] <= 0.05) & (df['entry_date'].dt.year == int(y)) #獲利介於0至5
    cond4 = (df['return'] > 0.1) & (df['entry_date'].dt.year == int(y)) #獲利大於 10%
    cond5 = (df['return'] > 0.3) & (df['entry_date'].dt.year == int(y)) #獲利大於 30%

    總獲利 = df[cond1]['return'].sum()
    總虧損 = df[cond2]['return'].sum()
    盈虧比 = abs(總獲利 / 總虧損)
    平均每筆獲利 = 總獲利 / len(df[cond1])
    平均每筆虧損 = 總虧損 / len(df[cond2])
    平均每筆盈虧比 = abs(平均每筆獲利 / 平均每筆虧損)

    yearlist.append(y)
    總獲利list.append(總獲利)
    總虧損list.append(總虧損)
    盈虧比list.append(盈虧比)
    平均每筆獲利list.append(平均每筆獲利)
    平均每筆虧損list.append(平均每筆虧損)
    平均每筆盈虧比list.append(平均每筆盈虧比)
    總交易筆數list.append(len(df[cond1]) + len(df[cond2]))
    獲利筆數list.append(len(df[cond1]))
    虧損筆數list.append(len(df[cond2]))
    勝率list.append(len(df[cond1])/(len(df[cond1]) + len(df[cond2])))

    v1 = round((len(df[cond3])/len(df[cond1])) * 100, 2)
    v2 = round((len(df[cond4])/len(df[cond1])) * 100, 2)
    v3 = round((len(df[cond5])/len(df[cond1])) * 100, 2)
    獲利介於0至5的筆數佔總獲利筆數list.append(v1)
    獲利大於10的筆數佔總獲利筆數list.append(v2)
    獲利大於30的筆數佔總獲利筆數list.append(v3)

cond1 = df['return'] > 0
cond2 = df['return'] < 0
cond3 = (df['return'] >= 0) & (df['return'] <= 0.05)
cond4 = df['return'] > 0.1
cond5 = df['return'] > 0.3
總獲利 = df[cond1]['return'].sum()
總虧損 = df[cond2]['return'].sum()
盈虧比 = abs(總獲利 / 總虧損)
平均每筆獲利 = 總獲利 / len(df[cond1])
平均每筆虧損 = 總虧損 / len(df[cond2])
平均每筆盈虧比 = abs(平均每筆獲利 / 平均每筆虧損)

yearlist.append('全部年度')
總獲利list.append(總獲利)
總虧損list.append(總虧損)
盈虧比list.append(盈虧比)
平均每筆獲利list.append(平均每筆獲利)
平均每筆虧損list.append(平均每筆虧損)
平均每筆盈虧比list.append(平均每筆盈虧比)
總交易筆數list.append(len(df[cond1]) + len(df[cond2]))
獲利筆數list.append(len(df[cond1]))
虧損筆數list.append(len(df[cond2]))
勝率list.append(len(df[cond1])/(len(df[cond1]) + len(df[cond2])))

v1 = round((len(df[cond3])/len(df[cond1])) * 100, 2)
v2 = round((len(df[cond4])/len(df[cond1])) * 100, 2)
v3 = round((len(df[cond5])/len(df[cond1])) * 100, 2)
獲利介於0至5的筆數佔總獲利筆數list.append(v1)
獲利大於10的筆數佔總獲利筆數list.append(v2)
獲利大於30的筆數佔總獲利筆數list.append(v3)

dic = {
  '年度': yearlist,
  '總獲利': 總獲利list,
  '總虧損': 總虧損list,
  '盈虧比': 盈虧比list,
  '平均每筆獲利': 平均每筆獲利list,
  '平均每筆虧損': 平均每筆虧損list,
  '平均每筆盈虧比': 平均每筆盈虧比list,
  '總交易筆數': 總交易筆數list,
  '獲利筆數': 獲利筆數list,
  '虧損筆數': 虧損筆數list,
  '勝率': 勝率list,
  '獲利介於0至5的筆數佔總獲利筆數(%)': 獲利介於0至5的筆數佔總獲利筆數list,
  '獲利大於10的筆數佔總獲利筆數(%)': 獲利大於10的筆數佔總獲利筆數list,
  '獲利大於30的筆數佔總獲利筆數(%)': 獲利大於30的筆數佔總獲利筆數list
}

result = pd.DataFrame(dic)
result

In [ ]:
report.display_mae_mfe_analysis()

#抬轎二

In [ ]:
from finlab import data
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import numpy as np

# 獲取數據
close = data.get('price:收盤價')
vol = data.get('price:成交股數')
rev = data.get('monthly_revenue:當月營收')
rev_yoy_growth = data.get('monthly_revenue:去年同月增減(%)')
每股盈餘 = data.get('financial_statement:每股盈餘')
cap = data.get('etl:market_value')

# 計算技術指標
rsi10 = data.indicator("RSI", timeperiod=10)
atr = data.indicator('ATR', adjust_price=True, timeperiod=10)
adj_close = data.get('etl:adj_close')
entry_volatility = atr / adj_close
natr60 = data.indicator('NATR', timeperiod=60)

# 計算益本比
益本比 = 每股盈餘 / close

# 融資使用率
融資使用率 = data.get('margin_transactions:融資使用率')



# 小投資者的持股比例（持股分級在6以內）
inventory = data.get("inventory")
small_inv = (inventory[inventory.持股分級.astype(int) <= 6]
             .reset_index()
             .groupby(["date", "stock_id"])
             .agg({"占集保庫存數比例": "sum"})
             .reset_index()
             .pivot(index="date", columns="stock_id")["占集保庫存數比例"] <= 24)

# 條件篩選
conditions = (
    FinlabDataFrame(small_inv)
    & ((rev.pct_change().rolling(6).mean() > 0).index_str_to_date())  # 最近6個月營收變動率為正
    & ((益本比 < 0.1) | (益本比.isna()))  # 益本比小於0.1或為NaN
    & (natr60.rank(axis=1, pct=True) < 0.7)  # NATR60排名在前70%
    & (close > close.rolling(120).mean())  # 收盤價大於120日均線
    & (close > close.rolling(20).mean())  # 收盤價大於20日均線
    & (entry_volatility <= 0.033)  # 進場波動率小於等於0.033
    & (cap < 1e10)  # 市值小於100億
    & (rsi10.pct_change(3) > 0.04)  # RSI10在3天內變化超過4%
    & (vol > 300 * 1000)  # 成交量大於300千股
    & (data.get('fundamental_features:稅後淨利成長率') > 0)  # 稅後淨利成長率大於0


)

# 設定持倉
position = rev_yoy_growth * conditions
position = position[position > 0].is_largest(6).reindex(rev.index_str_to_date().index, method='ffill')

# 回測策略
report = sim(position=position.loc["2017-01-01":],
             stop_loss=0.125,
             take_profit=0.8,
             position_limit=0.25,
             fee_ratio=1.425 / 1000 * 0.3,
             upload=True,
             trade_at_price='close',
             mae_mfe_window=40,
             live_performance_start='2024-02-01',
             name='抬轎二')


# 抬轎一加強版三

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest

rev =  data.get('monthly_revenue:當月營收')
rev_yoy_growth = data.get('monthly_revenue:去年同月增減(%)')
volume = data.get('price:成交股數')
close = data.get('price:收盤價')
margin_transactions = data.get('margin_transactions:融資今日餘額')
other_income = data.get('fundamental_features:業外收支營收率')
ope_earn = data.get('fundamental_features:營業利益率')
市值 = data.get('etl:market_value')
#high = data.get("price:最高價")
#low = data.get("price:最低價")
new_low = close == close.rolling(40).min()
no_new_low_days = new_low.rolling(2).sum() == 0

中軌 = close.rolling(window=40).mean()
標準差 = close.rolling(window=40).std()
上軌 = 中軌 + 1.5 * 標準差
#下軌 = 中軌 - 1.2 * 標準差

class BuyCondition:
    def __init__(self):
        self.revenue = data.get('monthly_revenue:當月營收')
        self.稅後淨利 = data.get('fundamental_features:經常稅後淨利')
        self.權益總計 = data.get('financial_statement:股東權益總額')
        self.close =  data.get("price:收盤價")
    def revenue_new_high(self,rev_ma=2,rolling_window=12,rolling_period=6):
        '''
定義:
    創新高營收
目的:
    公司創下營收新高通常是一個正面訊號，代表公司的業務成長和獲利能力提升。
    但是否能帶動股價上漲，還需要進一步分析其背後的原因和其他因素。

以下是幾個重點：

    營收成長的原因：
                需要判斷營收創高是源自於產業需求增加、公司競爭力提升，
                還是只是淡旺季波動或會計因素造成的短期現象。
                唯有反映公司長期成長趨勢的營收增長，才是真正的利多。
    獲利能力的變化：
                營收增加未必等於獲利增加，還要看毛利率和淨利率的變化。
                如果營收增長的同時獲利率下滑，對股價的助益有限。
    市場預期的差異：
                若營收增長低於市場預期，或者之前已有利多消息釋出而股價已反應，
                則營收創高未必能再帶動股價上揚。
    籌碼面的狀況：
                主力是否已經在營收公布前布局完成並準備出貨，也會影響股價走勢。
                如果籌碼已經調度到位，營收利多出盡後反而容易回檔。
    成交量的配合：
                若營收創高時成交量未能放大，缺乏買盤支撐，股價也難有可觀的表現。

    總之，營收創新高是值得關注的利多訊號，但投資人還是要審慎評估其品質和持續性，
    並綜合基本面、技術面、籌碼面等多方面因素，才能做出正確的投資判斷。
    光看營收數字而追高，反而可能落入主力的出貨陷阱。

        '''
        rev_ma = self.revenue.average(rev_ma)
        condition = (rev_ma ==
                     rev_ma.rolling(rolling_window, min_periods=rolling_period).max())
        return condition

    def return_on_equity(self):
        '''
定義:
    股東權益報酬率或淨值報酬率,是一個用來衡量公司獲利能力的重要財務指標。
目的:
    它表示股東投入的每一元淨值(權益),可以創造出多少淨利潤,反映公司運用股東資金的效率。
        '''

        return_on_equity = self.稅後淨利 / self.權益總計
        return return_on_equity

    def rsv(self, n=50) -> pd.DataFrame:
        rsv = ((self.close - self.close.rolling(n).min())
               / (self.close.rolling(n).max() - self.close.rolling(n).min()))
        return rsv

c = BuyCondition()


all_conditions = (


    (rev / rev.shift() > 0.8) &
    (rev.average(3).pct_change(6) > rev.average(12).pct_change(6)) &
    (~(rev_yoy_growth < 0).sustain(5,2)) &
    (c.revenue_new_high(rolling_window=12)) &
    (c.return_on_equity() > 0) &
    (市值 < 市值.quantile_row(0.8))&
    (close > close.average(5))&
    #(close>上軌)&
    no_new_low_days &
    (c.rsv(n=60).rank(pct=True, axis=1) > 0.55) &
    ((margin_transactions.rank(pct=True) < 0.3).sustain(5,3)) &
    (other_income < 5) &
    (ope_earn >= ope_earn.quantile_row(0.25))
)


turnover = data.get("price:成交金額").average(5)

position= turnover*all_conditions

final_position = position[position>0].is_largest(3).reindex(rev.index_str_to_date().index, method='ffill').fillna(False)



strategy_name='抬轎一加強版三'
report = sim(
            final_position
            ,stop_loss=0.11
            ,take_profit=0.4
            ,mae_mfe_window=22
            ,mae_mfe_window_step=2
            ,trade_at_price='close'
            ,fee_ratio=1.425/1000*0.18
            ,upload=True
            ,name=strategy_name
            ,live_performance_start='2024-05-01'
            )



In [ ]:
# 儲存csv檔
#report.get_trades().to_csv('/content/drive/MyDrive/抬轎一加強版三.csv')

In [ ]:
report.display_mae_mfe_analysis()

# 抬轎一加強版二

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest

市值 = data.get('etl:market_value')
volume = data.get("price:成交股數")
普通股股本 = data.get('financial_statement:普通股股本')
turnover_rate=(volume) /(普通股股本 /10)
rev = data.get('monthly_revenue:當月營收')
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
score = mrev_1m_yoyg.rank(pct=True, axis=1)>=0.55
close = data.get('price:收盤價')
融資今日餘額 = data.get("margin_transactions:融資今日餘額")

inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 14)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))


ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
    (close.notna()))


with data.universe(market="OTC"):
    otc_close = data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)

RS = ((close/close.shift(5)) / (benchmark/benchmark.shift(5)))

rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())

new_low = close == close.rolling(20).min()
no_new_low_days = new_low.rolling(4).sum() == 0



position= (
            (rev / rev.shift(12) > 1.05) &
            (score) &
            (市值 < 市值.quantile_row(0.8))&
            (close > close.average(5)) &
            (close > close.average(240)) &
            (close.average(6) > close.average(21))&
            (no_new_low_days) &
            (turnover_rate.average(10) > 0.8) &
            ((RS.rank(pct=True, axis=1) > 0.6))&
            ((s.rank(pct=True, axis=1) > 0.25))&
            ((data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.1).sustain(6,4))
            )


condition = position[position>0]
turnover = data.get("price:成交金額")
vol = data.get('price:成交股數').average(10)
position = (condition * vol).is_largest(30)

position = (position * turnover).is_largest(10)

buy = (position * rsv).is_largest(5).reindex(rev.index_str_to_date().index)

report = sim(
            buy
            ,stop_loss=0.1
            ,take_profit=0.4
            ,trade_at_price='close'
            ,fee_ratio=1.425/1000*0.18
            ,upload=True
            ,name='抬轎一加強版二'
            ,mae_mfe_window=22
            ,mae_mfe_window_step=2
            ,live_performance_start='2024-05-01'
            )

In [ ]:
report.display_mae_mfe_analysis()

# 抬轎一加強版一

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest


close = data.get('price:收盤價')
股本 = data.get('financial_statement:股本')
adj_close = data.get('etl:adj_close')
volume =  data.get("price:成交股數")
普通股股本 = data.get('financial_statement:普通股股本')
市值 = data.get('etl:market_value')
new_low = close == close.rolling(20).min()
no_new_low_days = new_low.rolling(4).sum() == 0


inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 15)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))


ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
    (close.notna()))


with data.universe(market="OTC"):
    otc_close =  data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)


def relative_strength(n=8):
    RS = ((close/close.shift(n)) / (benchmark/benchmark.shift(n)))
    return RS

def turnover_rate():
    return (volume) /(普通股股本 /10)

def entry_signal_volatility(timeperiod=5):

    atr = data.indicator('ATR',adjust_price=True,timeperiod=timeperiod)
    entry_volatility = atr/adj_close
    return entry_volatility


position = (
    (data.get('price:成交股數').average(5) > 100*1000) &
    (~(data.get('monthly_revenue:去年同月增減(%)') <0).sustain(5,2)) &
    (市值 < 市值.quantile_row(0.8))&
    (entry_signal_volatility(timeperiod=10) < 0.4) &
    (close > close.average(60)) &
    no_new_low_days &
    ((s.rank(pct=True, axis=1) > 0.35)) &
    (turnover_rate().rank(pct=True, axis=1) > 0.25).sustain(3,2) &
    (relative_strength().rank(pct=True, axis=1) > 0.55) &
    (data.get('margin_transactions:融資今日餘額').rank(pct=True) <0.3).sustain(5,4) &
    (data.indicator("RSI", timeperiod=10) < 90) &
    (data.get('fundamental_features:營業利益率') >= 2.5) &
    (data.get('fundamental_features:營業毛利率') >= 1.1)

    )
rev =  data.get('monthly_revenue:當月營收')
rev_yoy_growth = data.get('monthly_revenue:去年同月增減(%)')
position= rev_yoy_growth*position

position = position[position>0].is_largest(5).reindex(rev.index_str_to_date().index, method='ffill')

strategy_name = '抬轎一加強版一'

report = sim(
        position
        ,upload=True
        ,trade_at_price='close'
        ,stop_loss=0.1
        ,fee_ratio=1.425/1000*0.18
        ,name = strategy_name
        ,mae_mfe_window=22
        ,mae_mfe_window_step=2
        ,live_performance_start='2024-05-01'
            )

In [ ]:
report.display_mae_mfe_analysis()

# GA_yoy_v1

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest

adj_close = data.get('etl:adj_close')
cap = data.get('etl:market_value')
volume = data.get("price:成交股數")
普通股股本 = data.get('financial_statement:普通股股本')
turnover_rate=(volume) /(普通股股本 /10)
close = data.get('price:收盤價')





def close_volatility(timeperiod=20):
    close_volatility = close.pct_change().rolling(timeperiod).std().rank(axis=1, pct=True)
    return close_volatility

def entry_signal_volatility(timeperiod=5):

    atr = data.indicator('ATR',adjust_price=True,timeperiod=timeperiod)
    entry_volatility = atr/adj_close
    return entry_volatility




cond_all =(

      (cap<cap.quantile_row(0.8))&
      (close>close.average(60))&
      (entry_signal_volatility(timeperiod=10) < 0.4) &
      (data.get('fundamental_features:營業利益率')>3) &
      (close_volatility(timeperiod=20)<0.6) &
      (data.indicator('RSI',timeperiod=10) < 90) &
      ((data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.2).sustain(5,3))&
      (turnover_rate.average(5) > turnover_rate.average(20))
)


cond_all = cond_all & (data.get('price:成交股數').average(10) > 200*1000)

去年同月增減 = data.get("monthly_revenue:去年同月增減(%)")
cond_all = cond_all*去年同月增減
cond_all = cond_all[cond_all>0].is_largest(4)

strategy_name ="GA_yoy_v1"

report = sim(cond_all
                    ,resample='M'
                    ,resample_offset='12D'
                    ,trade_at_price='close'
                    ,fee_ratio=1.425/1000*0.18
                    ,take_profit=0.4
                    , stop_loss=0.1
                    # , position_limit=0.33
                    ,upload = True
                    ,name=strategy_name
                    ,mae_mfe_window=40
                    ,live_performance_start='2024-05-01'

                    )
report.display_mae_mfe_analysis()

# GA_yoy_v2

In [ ]:
import pandas as pd
from finlab import data, backtest
import numpy as np
from finlab.dataframe import FinlabDataFrame

# 獲取所需數據
rev = data.get('monthly_revenue:當月營收')
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
close = data.get('price:收盤價')
rev_yoy_growth = data.get('monthly_revenue:去年同月增減(%)')
df1 = data.get('financial_statement:投資活動之淨現金流入_流出')
df2 = data.get('financial_statement:營業活動之淨現金流入_流出')
vol = data.get("price:成交股數") / 1000
cap = data.get('etl:market_value')
# 計算各種技術指標
rev_ma4 = rev.rolling(4).mean()
rev_ma12 = rev.rolling(12).mean()
rsv = (close - close.rolling(120).min()) / (close.rolling(120).max() - close.rolling(120).min())
fcf = (df1 + df2).rolling(4).mean()
new_low = close == close.rolling(20).min()



# 計算燭台波動率
def compute_candle_volatility(timeperiod=20):
    close = data.get("price:收盤價")
    high = data.get("price:最高價")
    low = data.get("price:最低價")
    open_ = data.get("price:開盤價")

    bullish_candle = close >= open_
    bullish_volatility = abs(close.shift() - open_) + abs(open_ - low) + abs(low - high) + abs(high - close)
    bearish_volatility = abs(close.shift() - open_) + abs(open_ - high) + abs(high - low) + abs(low - close)

    candle_volatility = FinlabDataFrame(np.nan, index=close.index, columns=close.columns)
    candle_volatility[bullish_candle] = bullish_volatility
    candle_volatility[~bullish_candle] = bearish_volatility

    volatility = candle_volatility.rolling(timeperiod).mean() / close.rolling(timeperiod).mean() * 100
    return volatility

volatility = compute_candle_volatility()



# 條件篩選
cond_all = (

    (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.55) &
    (rev_ma4/rev_ma12>1) &
    (fcf > 0) &
    (rsv > 0.85) &
    (close > close.average(6)) &
    (close > close.average(20)) &
    (new_low.rolling(4).sum() == 0) &
    (vol > 150) &
    (volatility <= 8 ) &
    (cap < 1e10) &
    (cap > cap.quantile(0.1, axis=1))

)

# 選股結果
result = rev_yoy_growth * cond_all
buy = result[result > 0].is_largest(5).reindex(rev.index_str_to_date().index, method='ffill')

# 賣出條件
sell_1 = buy.reindex(rev.index_str_to_date().index - pd.DateOffset(days=3), method='ffill')
sell = sell_1

# 持股部位
position = buy.hold_until(sell)

# 策略名稱
strategy_name = 'GA_yoy_v2'

# 回測
report = backtest.sim(
    position=position,
    fee_ratio=1.425 / 1000 * 0.22,
    upload=True,
    name=strategy_name,
    trade_at_price='close',live_performance_start='2024-05-01'
)


# yoy_test

In [ ]:
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame



close = data.get('price:收盤價')
vol = data.get('price:成交股數')
rev = data.get('monthly_revenue:當月營收')
rev_ma = rev.average(2)
rev_yoy_growth = data.get('monthly_revenue:去年同月增減(%)')
cap = data.get('etl:market_value')

inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 14)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))


ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(8)).rank(axis=1, pct=True) *
    (close.notna()))


conditions = ((rev_ma == rev_ma.rolling(9, min_periods=6).max())
        & (close == close.rolling(200).max()).sustain(5,2)
        & (vol.average(5) > 500*1000)
        & (cap < cap.quantile_row(0.8))
        & ((s.rank(pct=True, axis=1) > 0.25))
              )

position= rev_yoy_growth*conditions
position = position[position>0].is_largest(5).reindex(rev.index_str_to_date().index, method='ffill')

# 設定position_limit避免重壓
report =sim(position=position,
    stop_loss=0.16,
    take_profit=0.8,
    position_limit=0.25,
    fee_ratio=1.425/1000*0.3,
    name="yoy_test",
    upload =True,
    mae_mfe_window=40,
    live_performance_start='2024-06-21')


#report.display_mae_mfe_analysis()

# yoy_test2

In [ ]:
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame



close = data.get('price:收盤價')
vol = data.get('price:成交股數')
rev = data.get('monthly_revenue:當月營收')
rev_ma = rev.average(2)
#rev_year_growth = data.get('monthly_revenue:去年同月增減(%)')

cap = data.get('etl:market_value')

inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 14)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))


ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(8)).rank(axis=1, pct=True) *
    (close.notna()))


conditions = ((rev_ma == rev_ma.rolling(9, min_periods=6).max())
        & (close == close.rolling(200).max()).sustain(5,2)
        & (vol.average(5) > 500*1000)
        & ((s.rank(pct=True, axis=1) > 0.25))
        & (data.get("etl:disposal_stock_filter"))
        & (data.get("etl:full_cash_delivery_stock_filter"))
              )



position= cap*conditions
position = position[position>0].is_smallest(5).reindex(rev.index_str_to_date().index, method='ffill')

# 設定position_limit避免重壓
report =sim(position=position,
    stop_loss=0.1,
    take_profit=0.8,
    position_limit=0.25,
    fee_ratio=1.425/1000*0.3,
    name="yoy_test2",
    upload =True,
    mae_mfe_window=40,
    trade_at_price='close',
    live_performance_start='2024-06-21')


#report.display_mae_mfe_analysis()

# operating_profit_ratio

In [ ]:
from finlab import data
from finlab.backtest import sim
import pandas as pd
import numpy as np
from finlab import backtest

rev = data.get('monthly_revenue:當月營收')
vol = data.get('price:成交股數')
operating_income = data.get('financial_statement:營業收入淨額')
operating_profit = data.get('financial_statement:營業利益')
close = data.get('price:收盤價')
cap = data.get('etl:market_value')
operating_profit_ratio = (operating_profit / operating_income) * 100
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())

position = (

    ((((operating_profit_ratio - operating_profit_ratio.shift(1)) / abs(operating_profit_ratio.shift(1))).rolling(3).min()) * 100 > 5)&
    (operating_profit_ratio.rolling(4).min() > 0)&
    (operating_profit_ratio.rise(4))&
    (cap < cap.quantile_row(0.8))&
    (vol.average(5) > 100*1000)

 )

position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)

# 設定position_limit避免重壓
report =sim(position,
    stop_loss=0.1,
    take_profit=0.8,
    position_limit=0.25,
    fee_ratio=1.425/1000*0.3,
    name="operating_profit_ratio",
    upload =True,
    mae_mfe_window=40,
    trade_at_price='close',
    live_performance_start='2024-06-21')

#report.display_mae_mfe_analysis()

# 忍者龜

In [ ]:
from finlab import data
from finlab.dataframe import FinlabDataFrame
from finlab import login
import numpy as np
import pandas as pd
import os
from finlab import backtest

close = data.get('price:收盤價')
vol = data.get('price:成交股數')
sma20 = close.average(20)
sma60 = close.average(60)
rev = data.get('monthly_revenue:當月營收')
rev_ma3 = rev.average(2)
rev_ma12 = rev.average(12)
ope_earn = data.get('fundamental_features:營業利益率')
yield_ratio = data.get('price_earning_ratio:殖利率(%)')
boss_hold = data.get("internal_equity_changes:董監持有股數占比")
去年同月增減 = data.get("monthly_revenue:去年同月增減(%)")
df = data.get('security_categories')

#刪除金融營建與ky股
sc = data.get("security_categories")
delete_stock = data.get('price:收盤價')
delete_stock_col = delete_stock.columns
ky_filter = delete_stock_col[~delete_stock_col.isin(list(sc[sc["name"].str.contains("萬泰科")]["stock_id"]))]
category_filter = delete_stock_col[~delete_stock_col.isin(list(sc[sc["category"].str.contains("鋼鐵|建材營造|連接器|太空衛星科技|金融")]['stock_id']))]
刪除特定股票= delete_stock[ky_filter.intersection(category_filter)]
cond7=df = 刪除特定股票
#new_low = close == close.rolling(20).min()
#no_new_low_days = new_low.rolling(3).sum() == 0
#pe = data.get('price_earning_ratio:本益比')

cond1 = yield_ratio >= 6
#cond1 = yield_ratio>=yield_ratio.quantile_row(0.5)
cond2 = (close > sma20) & (close > sma60)
cond3 = rev.average(3) > rev.average(12)
cond4 = ope_earn >= 3
cond5 = boss_hold >= 10
cond6 = (vol.average(5) >= 50*1000) & (vol.average(5) <= 400*1000)
cond_all = (cond1
            & cond2
            & cond3
            & cond4
            & cond5
            & cond6
            & cond7
            & no_new_low_days
            #& (pe<pe.quantile_row(0.5))


            )

cond_all = cond_all*去年同月增減
cond_all = cond_all[cond_all>0].is_largest(5)


report = backtest.sim(cond_all,
                        resample='M',
                        resample_offset='12D',
                        trade_at_price='open',
                        fee_ratio=1.425/1000*0.22,
                        stop_loss=0.1,
                        position_limit=0.33,
                        upload=True,
                        name = '高殖利率忍者龜'
                        )



# GA_peg_v1

In [ ]:
from finlab.backtest import sim
from finlab import data
import numpy as np
import pandas as pd
from finlab import dataframe
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import os
from finlab import backtest



volume = data.get("price:成交股數")
普通股股本 = data.get('financial_statement:普通股股本')
turnover_rate=(volume) /(普通股股本 /10)
close = data.get('price:收盤價')
gross_rate = data.get('fundamental_features:營業毛利率')
rev =  data.get('monthly_revenue:當月營收')
#mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100

def revenue_new_high(rev_ma=2,rolling_window=12,rolling_period=6):
    rev_ma = rev.average(rev_ma)
    condition = (rev_ma ==rev_ma.rolling(rolling_window, min_periods=rolling_period).max())
    return condition


cond_all = (
(revenue_new_high(rolling_window=9)) &
(rev.average(3).pct_change(12) > rev.average(12).pct_change(12))&
(close>close.average(10))&
(volume >= volume.rolling(60).mean() * 0.8)&
(volume>600*1000)&
(gross_rate>gross_rate.quantile_row(0.2))&
((turnover_rate.average(5)) > turnover_rate.average(20))&
((turnover_rate.average(5)) > turnover_rate.average(60))&
(data.indicator("RSI", timeperiod=10) > 20) &
(data.indicator("RSI", timeperiod=10) < 90) &
((data.get('margin_transactions:融資今日餘額').rank(pct=True) < 0.4).sustain(5,4))&
(data.get('price:成交股數').average(10) > 100*1000)

)

營業利益成長率=data.get('fundamental_features:營業利益成長率')
pe = data.get('price_earning_ratio:本益比')
peg=(pe/營業利益成長率)

result=peg*cond_all
position=result[result>0].is_smallest(5).reindex(rev.index_str_to_date().index, method='ffill')

strategy_name ="GA_peg_v1"



report= sim(
      position=position
      ,name = strategy_name
      ,fee_ratio=1.425/1000*0.18
      # ,stop_loss = 0.1
      ,take_profit = 0.4
      ,trade_at_price='open_close_avg'
      ,upload=True
      ,mae_mfe_window=40
      )



# 黑皮

In [ ]:
from finlab import data
from finlab.dataframe import FinlabDataFrame
from finlab import login
import numpy as np
import pandas as pd
import os
from finlab.backtest import sim


def make_total_iterations(**params):
    return '_'.join(f'{k}_{v}' for k, v in params.items())

class BuyCondition:

    def __init__(self):
        login(os.getenv('finlab_id_token'))
        data.set_storage(data.FileStorage())
        self.close =  data.get("price:收盤價")
        self.volume =  data.get("price:成交股數")
        self.inv = data.get('inventory')
        self.close = data.get("price:收盤價")
        self.股本 = data.get('financial_statement:股本')
        self.revenue = data.get('monthly_revenue:當月營收')
        self.tse_par = data.get('par_value_change_tse:twse_par_value_change_divide_ratio')
        self.otc_par = data.get('par_value_change_otc:otc_par_value_change_divide_ratio')


    def all_time_high_revenue(self,rev_ma=2,rolling_window=12,rolling_period=6):
        rev_ma = self.revenue.average(rev_ma)
        condition = (rev_ma ==
                     rev_ma.rolling(rolling_window, min_periods=rolling_period).max())
        return condition


    def small_inv(self,level = 7,datatype = '持有股數'):

        small_inv = FinlabDataFrame(self.inv[self.inv.持股分級.astype(int) <= level]
                                .reset_index().groupby(['date', 'stock_id'])
                                .agg({datatype: 'sum'})
                                .reset_index()
                                .pivot(index = 'date',columns= 'stock_id')[datatype])
        return small_inv

    def boss_inv(self,down_level = 9,  up_level= 15 ,datatype = '持有股數'):

        boss_inv =  FinlabDataFrame(
            self.inv[(self.inv.持股分級.astype(int) >= down_level)
                    & (self.inv.持股分級.astype(int) <= up_level)]
                                    .reset_index()
                                    .groupby(['date', 'stock_id'])
                                    .agg({datatype: 'sum'})
                                    .reset_index()
                                    .pivot(index='date',columns= 'stock_id')[datatype])
        return boss_inv



class SellCondition():
    def __init__(self):
        login(os.getenv('finlab_id_token'))
        data.set_storage(data.FileStorage())
        self.close =  data.get("price:收盤價")
    def time_offset(self,buy,simple_df,offset=3):
        sell = buy.reindex(simple_df.index_str_to_date()
                           .index-pd.DateOffset(days=offset), method='ffill')

        union_index = self.close.index.union(sell.index)
        sell = sell.reindex(union_index).fillna(False)
        buy = buy.reindex(union_index).fillna(False)
        buy = buy.reindex(union_index).fillna(False)

        return buy,sell


category_filter = [
  '鋼鐵工業','金融','水泥工業','橡膠工業','塑膠工業',
  '光電業', '其他', '其他電子業', '化學工業', '半導體',
  '存託憑證', '建材營造', '文化創意業',  '汽車工業',
  '油電燃氣業', '玻璃陶瓷', '生技醫療', '生技醫療業', '紡織纖維', '航運業',
  '觀光事業', '貿易百貨', '資訊服務業', '農業科技', '通信網路業', '造紙工業',
  '電器電纜', '電子商務', '電子通路業', '電子零組件', '電機機械',
  '電腦及週邊', '食品工業'
]
with data.universe(category=category_filter):

    營業利益成長率=data.get('fundamental_features:營業利益成長率')
    pe = data.get('price_earning_ratio:本益比')
    rev=data.get('monthly_revenue:當月營收')
    close=data.get('price:收盤價')
    vol = data.get("price:成交股數")
    市值 = data.get('etl:market_value')
    普通股股本 = data.get('financial_statement:普通股股本')
    turnover_rate=(vol) /(普通股股本 /10)
    c = BuyCondition()


    h1 = c.small_inv(level = 5,datatype = '持有股數')
    h2 = c.boss_inv(down_level = 9,  up_level= 15,datatype = '持有股數')
    ratio = (h2 / (h1 + h2))
    ratio2 = (h1 / (h1 + h2))
    peg=(pe/營業利益成長率)
    rev_ma3=rev.average(3)
    rev_ma12=rev.average(12)
   # rev_ma13=rev.average(13)

    p1 = (FinlabDataFrame((ratio).diff(2)) *
        (close.notna()))

    p2 = (FinlabDataFrame((ratio2).diff(2)) *
    (close.notna()))


    s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
        (close.notna()))

    #近期創低
    new_low = close == close.rolling(10).min()
    #近二日未創低
    no_new_low_3days = new_low.rolling(3).sum() == 0

    rs = (close / close.shift(11) - 1).rank(axis=1, pct=True) * 100

    cond_all=(
            (rev_ma3/rev_ma12>1.05) #營收成長
            &(rev/rev.shift(12)>0.95) #營收成長
            & (rs > rs.quantile_row(0.45))

            & (p2 < 0) #散戶減少
            & (close<close.quantile_row(0.7)) #價格上限,漲太高就不管了
            & (c.all_time_high_revenue(rev_ma=2,
                                    rolling_window=25,
                                    rolling_period=6
                                    )) #營收創新高
            & (s.rank(pct=True, axis=1) > 0.6) #大戶佔六成
            & (市值>市值.quantile_row(0.1))
            & (市值<市值.quantile_row(0.7))
            & (close>close.average(20))
            & (close>close.average(250))
            & no_new_low_3days
            &(close / close.shift(5) < 1.15)
            #& (boss_hold>=5)

            )


    result=peg*cond_all
    buy=result[result>0].is_smallest(4).reindex(rev.index_str_to_date().index, method='ffill')


    s = SellCondition()
    simple_df = rev.index_str_to_date()
    buy,sell = s.time_offset(buy,simple_df,offset=3)

    position = buy.hold_until(sell)

    report= sim(
            position=position.loc["2017-01-01":],
            fee_ratio=1.425/1000*0.22,
            trade_at_price='open',
            stop_loss=0.12,
            position_limit=0.4,
            upload=True,
            mae_mfe_window=40,
            name='黑皮'
            )



In [ ]:
# 儲存csv檔
#report.get_trades().to_csv('/content/drive/MyDrive/黑皮.csv')

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#max_trade_index = report.get_trades().index.max()
#print("最大交易索引:", max_trade_index)

# 抬轎

In [ ]:
from finlab import data
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import numpy as np
import pandas as pd
import os

# 技術指標
def compute_candle_volatility(timeperiod=20):
    close = data.get("price:收盤價")
    high = data.get("price:最高價")
    low = data.get("price:最低價")
    open_ =  data.get("price:開盤價")

    bullish_candle = close >= open_
    bullish_volatility = abs(close.shift() - open_) + abs(open_ - low) + abs(low - high) + abs(high - close)
    bearish_volatility = abs(close.shift() - open_) + abs(open_ - high) + abs(high-low) + abs(low - close)
    candle_volatility = FinlabDataFrame(np.nan, index=close.index, columns=close.columns)
    candle_volatility[bullish_candle] = bullish_volatility
    candle_volatility[~bullish_candle] = bearish_volatility
    volatility = candle_volatility.average(timeperiod) / close.average(timeperiod) * 100
    return volatility

volatility = compute_candle_volatility()

close= data.get('price:收盤價')
inv = data.get('inventory')
h1 = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 5]
      .reset_index().groupby(['date', 'stock_id'])
      .agg({'持有股數': 'sum'})
      .reset_index()
      .pivot(index='date', columns='stock_id', values='持有股數'))


h2 = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 11) &
            (inv.持股分級.astype(int) <= 15)]
            .reset_index().groupby(['date', 'stock_id'])
            .agg({'持有股數': 'sum'})
            .reset_index().pivot(index='date', columns='stock_id', values='持有股數'))

ratio = (h2 / (h1 + h2))

s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
    (close.notna()))

cap = data.get('etl:market_value')
vol = data.get('price:成交股數')
df1 = data.get('financial_statement:投資活動之淨現金流入_流出')
df2 = data.get('financial_statement:營業活動之淨現金流入_流出')
稅後淨利 = data.get('fundamental_features:經常稅後淨利')
權益總計 = data.get('financial_statement:股東權益總額')
稅後淨利成長率=data.get('fundamental_features:稅後淨利成長率')
high = data.get("price:最高價")
自由現金流 = (df1 + df2).rolling(4).mean()
股東權益報酬率 = 稅後淨利/權益總計
rev = data.get('monthly_revenue:當月營收')* 1000
#mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100

當季營收 = rev.rolling(4).sum()
市值營收比 = cap / 當季營收
rev_ma3 = rev.average(3)
rev_ma11 = rev.average(11)
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())
new_low = close == close.rolling(20).min()
no_new_low_2days = new_low.rolling(2).sum() == 0
short_volatility = (compute_candle_volatility(timeperiod=10))
long_volatility = (compute_candle_volatility(timeperiod=60))

with data.universe(market="OTC"):
    otc_close =  data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)

RS = ((close/close.shift(8)) / (benchmark/benchmark.shift(8)))

position = (
    (自由現金流 > 0)
    & (股東權益報酬率 > 0)
    & (稅後淨利成長率> 0)
    & (市值營收比 < 3 )
    & (cap<cap.quantile_row(0.75))
    & (cap>cap.quantile_row(0.1))
    & ((s.rank(pct=True, axis=1) > 0.3))
    & (rev_ma3/rev_ma11>0.95)
    & (rev/rev.shift(12)>1.05)
    & (volatility <= 7.5)
    & (vol > 150 * 1000)
    & (close.average(5)>close.average(90))
    & no_new_low_2days
    & ((RS.rank(pct=True, axis=1) > 0.5))
    )


high_rolling1 = high.rolling(10).max()
high_rolling2 = high.rolling(20).max()
sell_cond1 = close < high_rolling1*0.9
sell_cond2 = (close < high_rolling2 - 3*long_volatility) & (short_volatility/close.average(10)*100 > 6)
sell_cond3 =  close.rolling(10).mean() < close.rolling(20).mean()*0.9

sell_condition = sell_cond1 | sell_cond2 | sell_cond3
position[sell_condition] = False
position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)

#line_access_token = ["hxlnkinXYMD5gGHOTOSqfBhSULSLNcW6Z6Sb7z6Gf4R"]
report = sim(
            position,
            stop_loss=0.11,
            position_limit=0.3,
            trade_at_price='close',
            upload=True,
            name='抬轎',
            fee_ratio=1.425/1000*0.22,
            mae_mfe_window=40,live_performance_start='2023-09-01',
  					#notification_enable=True, line_access_token='hxlnkinXYMD5gGHOTOSqfBhSULSLNcW6Z6Sb7z6Gf4R'
            )
import matplotlib.pyplot as plt
from finlab import data, backtest

# Set the matplotlib style
plt.style.use('https://github.com/dhaitz/matplotlib-stylesheets/raw/master/pitayasmoothie-dark.mplstyle')
#plt.style.use('https://github.com/kimichenn/nord-deep-mpl-stylesheet/raw/main/nord-deep.mplstyle')

# Get financial data
close = data.get('price:收盤價')
vol = data.get('price:成交股數')

# Define Bollinger Bands calculations
中軌 = close.rolling(window=40).mean()
標準差 = close.rolling(window=40).std()
上軌 = 中軌 + 1.2 * 標準差
下軌 = 中軌 - 1.2 * 標準差

# Get stock IDs from the position DataFrame (where there are holdings)
stock_ids = position.columns[position.iloc[-1]]

# Determine the number of subplots
num_plots = len(stock_ids)
num_cols = min(num_plots, 4)  # Limit to 4 columns
num_rows = (num_plots + num_cols - 1) // num_cols  # Calculate the necessary number of rows

# Create a figure and a set of subplots
fig, axs = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6*num_cols, 6*num_rows))
axs = axs.flatten()  # Flatten the 2D array of axes for easy iteration

# Plot Bollinger Bands and Close Price for all stocks
for i, stock_id in enumerate(stock_ids):
    ax = axs[i]

    ax.plot(上軌[stock_id].loc['2024'], label='Upper Band')
    ax.plot(中軌[stock_id].loc['2024'], label='Middle Band')
    ax.plot(下軌[stock_id].loc['2024'], label='Lower Band')
    ax.plot(close[stock_id].loc['2024'], label='Close Price')

    ax.set_title(f'Bollinger Bands and Close Price for {stock_id}')
    ax.legend(loc='upper left')

    # Plot volume on the same axis but with a secondary y-axis
    ax2 = ax.twinx()
    ax2.bar(vol[stock_id].loc['2024'].index, vol[stock_id].loc['2024'], color='gray', alpha=0.3)
    ax2.set_ylabel('Volume')

# Hide any unused subplots
for j in range(i+1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.show()


In [ ]:
#(report.position!=0).sum(axis=1).plot()
#import pandas as pd
#pd.Series(report.get_stats())

In [ ]:
max_trade_index = report.get_trades().index.max()
print("最大交易索引:", max_trade_index)

# 暴力營收

In [ ]:
from finlab import data, backtest
from finlab.backtest import sim
import numpy as np
from finlab.dataframe import FinlabDataFrame


c = data.get('price:收盤價')
v = data.get("price:成交股數") / 1000
turnover = data.get('price:成交金額')
rev = data.get('monthly_revenue:當月營收')
rev = data.get('monthly_revenue:當月營收')
mrev_1m_yoyg = (rev - rev.shift(12)) / abs(rev.shift(12).where(rev.shift(12)!=0)) * 100
yoy = data.get('monthly_revenue:去年同月增減(%)')
mom = data.get('monthly_revenue:上月比較增減(%)')
cyoy = data.get('monthly_revenue:前期比較增減(%)')
margin_transaction_ratio = data.get('margin_transactions:融資使用率').fillna(0)
disposal_stocks = data.get('etl:disposal_stock_filter')
adj_close = data.get('etl:adj_close')


def rsv(n):
    return (c-c.rolling(n,int(n/2)).min()) / (c.rolling(n,int(n/2)).max()-c.rolling(n,int(n/2)).min())

def ma(n):
    return c.average(n)

def entry_signal_volatility(timeperiod=5):

    atr = data.indicator('ATR',adjust_price=True,timeperiod=timeperiod)
    entry_volatility = atr/adj_close
    return entry_volatility


with data.universe(market="OTC"):
    otc_close = data.get("price:收盤價")
    benchmark = otc_close.sum(axis=1)

RS1 = ((c/c.shift(5)) / (benchmark/benchmark.shift(5)))



c1 = v.average(10) > 500
c2 = disposal_stocks
c3 = margin_transaction_ratio < 34
c4 = yoy.rank(pct=True,axis=1) > 0.9
c4 = (mrev_1m_yoyg.rank(pct=True, axis=1)>=0.9)
c5 = cyoy.rank(pct=True,axis=1)> 0.5
c6 = (ma(6)>ma(21))
c7 = (c>ma(5))
c8 = ((RS1.rank(pct=True, axis=1) > 0.6))

position = (c1 & c2 & c3 & c4  & c5  & c6 & c7 & c8 )

position = (position[position>0] * rsv(50)).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)


report = sim(position,
        trade_at_price='close',
        name='暴力營收',
        upload=True,
        stop_loss=0.11,
        position_limit=0.3,
        fee_ratio=1.425 / 1000 * 0.22,
        live_performance_start='2024-06-20',
        mae_mfe_window=40)
#report.display_mae_mfe_analysis()

# 績效分析

In [ ]:
from finlab.plot import StrategySunburst

# 實例化物件
strategies = StrategySunburst()
strategies.plot(select_strategy={'抬轎':0.2,'抬轎一':0.2,'抬轎一加強版一':0.2,'抬轎一加強版二':0.2,'抬轎一加強版三':0.2},path =  [ 'category','stock_id','s_name']).show()

In [ ]:
from finlab.plot import StrategyReturnStats

start_date = '2024-01-01'
end_date  = '2024-12-31'

# 選定策略範圍
strategy_names = (['抬轎','抬轎一','抬轎一加強版一','抬轎一加強版二','抬轎一加強版三'])

report = StrategyReturnStats(start_date ,end_date, strategy_names)
# 繪製策略報酬率近期報酬率長條圖

report.plot_strategy_last_return().show()
report.plot_strategy_creturn().show()

# 麒麟

In [ ]:
from finlab import data
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import numpy as np


# 技術指標
def compute_candle_volatility(timeperiod=20):
    close = data.get("price:收盤價")
    high = data.get("price:最高價")
    low = data.get("price:最低價")
    open_ =  data.get("price:開盤價")

    bullish_candle = close >= open_
    bullish_volatility = abs(close.shift() - open_) + abs(open_ - low) + abs(low - high) + abs(high - close)
    bearish_volatility = abs(close.shift() - open_) + abs(open_ - high) + abs(high-low) + abs(low - close)
    candle_volatility = FinlabDataFrame(np.nan, index=close.index, columns=close.columns)
    candle_volatility[bullish_candle] = bullish_volatility
    candle_volatility[~bullish_candle] = bearish_volatility
    volatility = candle_volatility.average(timeperiod) / close.average(timeperiod) * 100
    return volatility

volatility = compute_candle_volatility()

close= data.get('price:收盤價')
#close_adj = data.get("etl:adj_close")
cap = data.get('etl:market_value')
vol = data.get('price:成交股數')
vol_ma = vol.average(10)
市值 = data.get('etl:market_value')


# 籌碼指標
inv = data.get("inventory")
retail_roportion = FinlabDataFrame(inv[inv.持股分級.astype(int) <= 4].
                                   reset_index().
                                   groupby(['date', 'stock_id']).
                                   agg({'占集保庫存數比例': 'sum'}).
                                   reset_index().pivot(columns='stock_id', index='date',values='占集保庫存數比例'))
boss_roportion = FinlabDataFrame(inv[(inv.持股分級.astype(int) >= 12) & (inv.持股分級.astype(int) <= 15)].
                                 reset_index().groupby(['date', 'stock_id']).
                                 agg({'占集保庫存數比例': 'sum'}).
                                 reset_index().
                                 pivot(columns='stock_id', index='date',values='占集保庫存數比例'))

# 基本面指標
rev = data.get('monthly_revenue:當月營收')
rev_ma = rev.average(2)
rev_year_growth = data.get('monthly_revenue:去年同月增減(%)')
#boss_hold = data.get("internal_equity_changes:董監持有股數占比")
#mon_rev_lag =data.get("monthly_revenue:去年當月營收")
#融資使用率=data.get('margin_transactions:融資使用率')
## 技術面條件
# 5日內有至少1日的股價創近100日新高
cond1 = (close == close.rolling(100).max()).sustain(5,2)
# 均線多頭排列
cond2 = close.average(5)>close.average(20)
# 5日均量大於200張
cond3 = vol_ma > 200*1000
# k線波動率低於12%，控制下檔波動
cond4 = volatility <= 11
## 籌碼條件
# 10張以下持股分級持股比例小於30%
cond5 = retail_roportion <= 27
# 10張以下持股分級持股比例較前4週下降
cond6 = retail_roportion < retail_roportion.shift(4)
# 400張以上持股分級持股比例較前4週增加
cond7 = boss_roportion > boss_roportion.shift(4)
# 規模因子，買中小型，股本小於500000張
cond8 = (市值 < 市值.quantile_row(0.8))
## 基本面條件
# 近2月平均營收創12期來新高
cond9 = rev_ma == rev_ma.rolling(24, min_periods=6).max()
# 排除月營收連3月衰退10%以上
cond10 = ~(rev_year_growth < 0).sustain(3)

position =( cond1
            & cond2
            & cond3
            & cond4
            & cond5
            & cond6
            & cond7
            & cond8
            & cond9
            & cond10
                      )
position *= close
position = position[position > 0]
position = position.is_smallest(5)
report = sim(position,
        resample="W",
        position_limit=1/4,
        mae_mfe_window=40,
        stop_loss=0.08,
        name='麒麟',
        upload=True,
        fee_ratio=1.425/1000/3
             )


In [ ]:
"""
from finlab import backtest

eps = data.get('financial_statement:每股盈餘')
roe = data.get('fundamental_features:ROE稅後')
eps_per_price = data.get('financial_statement:每股盈餘')	/ data.get('price:收盤價')





r = {}


r['毛'] = backtest.sim(毛.is_largest(30), upload=False)
r['毛'].creturn.plot(label='毛')

r['eps'] = backtest.sim(eps.is_largest(30), upload=False)
r['eps'].creturn.plot(label='eps')

r['roe'] = backtest.sim(roe.is_largest(30), upload=False)
r['roe'].creturn.plot(label='roe')

r['eps_per_price'] = backtest.sim(eps_per_price.is_largest(30), upload=False)
r['eps_per_price'].creturn.plot(label='better_eps')

r['roe_per_price'] = backtest.sim(roe_per_price.is_largest(30), upload=False)
r['roe_per_price'].creturn.plot(label='better_roe')

r['毛毛'] = backtest.sim(毛毛.is_largest(30), upload=False)
r['毛毛'].creturn.plot(label='better_毛毛')

import matplotlib.pyplot as plt
plt.legend()

In [ ]:
'''
from finlab import data
from finlab.backtest import sim

pe = data.get('price_earning_ratio:本益比')
rev=data.get('monthly_revenue:當月營收')
rev_ma3=rev.average(3)
rev_ma12=rev.average(12)
營業利益成長率=data.get('fundamental_features:營業利益成長率').deadline()
peg=(pe/營業利益成長率)



融資使用率=data.get('margin_transactions:融資使用率')
atr = data.indicator('ATR', adjust_price=True,timeperiod=10)
adj_close = data.get('etl:adj_close')
entry_volatility = atr/adj_close


cond_all=(
           (rev_ma3/rev_ma12>1.1)
          &(rev/rev.shift(1)>0.9)
          &(融資使用率 <= 25.875)
          &(entry_volatility <= 0.031)
          &(data.get('fundamental_features:業外收支營收率')<=5.2)

          )

result=peg*(cond_all)

position_upgrade=result[result>0].is_smallest(10).reindex(rev.index_str_to_date().index, method='ffill')
report_upgrade = sim(position=position_upgrade,
                     name="黑皮2",
                     fee_ratio=1.425/1000/3,
                     upload=True,
                     mae_mfe_window=30,
                     position_limit=0.05,stop_loss=0.3)

In [ ]:
"""
from finlab import data
import pandas as pd

# 獲取資料
dividend_announcement = data.get('dividend_announcement')
close_prices = data.get('dividend_tse:除權息參考價')
close_rights_prices = data.get('dividend_otc:除權息參考價')
close = data.get('price:最高價')
#data.get('price:收盤價')

# 篩選所需的欄位
selected_columns = [
    'stock_id',
    '公告日期',
    '除權交易日',
    '盈餘分配之股東現金股利(元/股)',
    '除息交易日',
    '法定盈餘公積、資本公積發放之現金(元/股)'
]

# 選取所需的資料
filtered_data = dividend_announcement[selected_columns]

# 將 close_prices 和 close_rights_prices 轉換為 dataframe 並將日期轉換為 datetime 格式
close_prices_df = close_prices.stack().reset_index()
close_prices_df.columns = ['date', 'stock_id', '除息參考價']

close_rights_prices_df = close_rights_prices.stack().reset_index()
close_rights_prices_df.columns = ['date', 'stock_id', '除權參考價']

# 確保日期欄位是 datetime 格式
filtered_data['除息交易日'] = pd.to_datetime(filtered_data['除息交易日'])
filtered_data['除權交易日'] = pd.to_datetime(filtered_data['除權交易日'])
close_prices_df['date'] = pd.to_datetime(close_prices_df['date'])
close_rights_prices_df['date'] = pd.to_datetime(close_rights_prices_df['date'])

# 確保 stock_id 的類型是一致的
filtered_data['stock_id'] = filtered_data['stock_id'].astype(str)
close_prices_df['stock_id'] = close_prices_df['stock_id'].astype(str)
close_rights_prices_df['stock_id'] = close_rights_prices_df['stock_id'].astype(str)

# 合併資料，確保 '除息交易日' 對應 'date'
merged_data = filtered_data.merge(close_prices_df, how='left', left_on=['stock_id', '除息交易日'], right_on=['stock_id', 'date'])
merged_data.drop(columns=['date'], inplace=True)

# 合併資料，確保 '除權交易日' 對應 'date'
merged_data = merged_data.merge(close_rights_prices_df, how='left', left_on=['stock_id', '除權交易日'], right_on=['stock_id', 'date'], suffixes=('', '_除權'))
merged_data.drop(columns=['date'], inplace=True)

# 確保收盤價資料的日期格式
close.index = pd.to_datetime(close.index)

def calculate_days_to_recover(stock_id, ex_date):
    if pd.isnull(ex_date) or stock_id not in close.columns:
        return None, None
    # 確保參考價格日期是前一天
    reference_price_date = ex_date - pd.Timedelta(days=1)
    if reference_price_date not in close.index:
        return None, None
    reference_price = close.loc[reference_price_date, stock_id]

    # 從除息日開始計算
    prices_after_event = close[stock_id].loc[ex_date:].dropna()
    if prices_after_event.empty:
        return None, None
    # 找到大於等於參考價的第一天
    for date, price in prices_after_event.items():
        if price >= reference_price:
            days_to_recover = (date - ex_date).days - 1  # 减少一天
            recovery_date = date
            return days_to_recover, recovery_date
    return None, None

# 計算填息花費日數和填息完成日期
merged_data[['填息花費日數', '填息完成日期']] = merged_data.apply(lambda row: calculate_days_to_recover(row['stock_id'], row['除息交易日']), axis=1, result_type='expand')

# 計算填權花費日數和填權完成日期
merged_data[['填權花費日數', '填權完成日期']] = merged_data.apply(lambda row: calculate_days_to_recover(row['stock_id'], row['除權交易日']), axis=1, result_type='expand')

# 篩選出 stock_id 為 1717 的行
result = merged_data[merged_data['stock_id'] == '1717']
# 顯示結果
print(result)

result = merged_data[merged_data['stock_id'] == '3663']
print(result)
"""

In [ ]:
"""from finlab import data
收盤價 = data.get("price:收盤價")
成交股數 = data.get("price:成交股數")
普通股股本 = data.get("financial_statement:普通股股本")
五日週轉率 = (成交股數.average(5)/普通股股本).iloc[-1]
排名前五日周轉率=五日週轉率.sort_values(ascending=False).head(20)
排名前五日周轉率

In [ ]:
'''
from finlab import data
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
import numpy as np
import pandas as pd
import os
data.set_storage(data.FileStorage())
from finlab import login


def make_total_iterations(**params):
    return '_'.join(f'{k}_{v}' for k, v in params.items())

class BuyCondition:

    def __init__(self):
        login(os.getenv('finlab_id_token'))
        data.set_storage(data.FileStorage())
        self.close =  data.get("price:收盤價")
        self.high =  data.get("price:最高價")
        self.low =  data.get("price:最低價")
        self.open =   data.get("price:開盤價")
        self.volume =  data.get("price:成交股數")
        # self.broker = data.get('broker_transactions')
        self.inv = data.get('inventory')
        self.close = data.get("price:收盤價")
        self.revenue = data.get('monthly_revenue:當月營收')
        self.營業利益成長率 = data.get('fundamental_features:營業利益成長率')
        self.股本 = data.get('financial_statement:股本')
        self.tse_par = data.get('par_value_change_tse:twse_par_value_change_divide_ratio')  # noqa: E501
        self.otc_par = data.get('par_value_change_otc:otc_par_value_change_divide_ratio')  # noqa: E501

    def capitalization(self):
        #市值上限
        empty_df = pd.DataFrame(1, columns=self.close.columns, index=self.close.index)
        tse_divide_ratio = (empty_df*self.tse_par).fillna(1).cumprod()
        otc_divide_ratio = (empty_df*self.otc_par).fillna(1).cumprod()
        par_divide_ratio = tse_divide_ratio*otc_divide_ratio
        市值 = self.股本 * self.close / 10 * 1000 * par_divide_ratio
        return 市值

    def max_market_capitalization(self,quantile=0.6):
        #市值上限
        empty_df = pd.DataFrame(1, columns=self.close.columns, index=self.close.index)
        tse_divide_ratio = (empty_df*self.tse_par).fillna(1).cumprod()
        otc_divide_ratio = (empty_df*self.otc_par).fillna(1).cumprod()
        par_divide_ratio = tse_divide_ratio*otc_divide_ratio
        市值 = self.股本 * self.close / 10 * 1000 * par_divide_ratio
        condition = 市值 < 市值.quantile_row(quantile)
        return condition

    def min_market_capitalization(self,quantile=0.1):
        #市值下限
        empty_df = pd.DataFrame(1, columns=self.close.columns, index=self.close.index)
        tse_divide_ratio = (empty_df*self.tse_par).fillna(1).cumprod()
        otc_divide_ratio = (empty_df*self.otc_par).fillna(1).cumprod()
        par_divide_ratio = tse_divide_ratio*otc_divide_ratio
        市值 = self.股本 * self.close / 10 * 1000 * par_divide_ratio
        condition = 市值 > 市值.quantile_row(quantile)
        return condition


    def closing_price_below_market_classification(self,quantile=0.9):
        #低價股
         # 收盤價低於整體市場分級的40%
        condition = self.close <= self.close.quantile_row(quantile)
        return condition

    def closing_price_uper_market_classification(self,quantile=0.1):
        #高價股
        # 收盤價高於整體市場分級的10%
        condition = self.close >= self.close.quantile_row(quantile)
        return condition

    def small_inv(self,level = 5,datatype = '持有股數'):

        small_inv = FinlabDataFrame(self.inv[self.inv.持股分級.astype(int) <= level]
                                .reset_index().groupby(['date', 'stock_id'])
                                .agg({datatype: 'sum'})
                                .reset_index()
                                .pivot(index = 'date',columns= 'stock_id')[datatype])
        return small_inv

    def boss_inv(self,down_level = 9,  up_level= 15 ,datatype = '持有股數'):

        boss_inv =  FinlabDataFrame(
            self.inv[(self.inv.持股分級.astype(int) >= down_level)
                    & (self.inv.持股分級.astype(int) <= up_level)]
                                    .reset_index()
                                    .groupby(['date', 'stock_id'])
                                    .agg({datatype: 'sum'})
                                    .reset_index()
                                    .pivot(index='date',columns= 'stock_id')[datatype])
        return boss_inv

    def compute_candle_volatility(self,timeperiod=20):


        bullish_candle = self.close >= self.open

        bullish_volatility = (
                                abs(self.close.shift() - self.open)
                                + abs(self.open - self.low)
                                + abs(self.low - self.high)
                                + abs(self.high - self.close)
                            )

        bearish_volatility = (abs(self.close.shift() - self.open)
                              + abs(self.open - self.high)
                              + abs(self.high-self.low)
                              + abs(self.low - self.close))

        candle_volatility = FinlabDataFrame(np.nan
                                            , index=self.close.index
                                            , columns=self.close.columns)


        candle_volatility[bullish_candle] = bullish_volatility

        candle_volatility[~bullish_candle] = bearish_volatility

        volatility = (candle_volatility.average(timeperiod)
                      / self.close.average(timeperiod) * 100)

        return volatility

股本 = data.get('financial_statement:股本')
close= data.get('price:收盤價')
vol = data.get('price:成交股數')
df1 = data.get('financial_statement:投資活動之淨現金流入_流出')
df2 = data.get('financial_statement:營業活動之淨現金流入_流出')
tse_par = data.get('par_value_change_tse:twse_par_value_change_divide_ratio')
otc_par = data.get('par_value_change_otc:otc_par_value_change_divide_ratio')
稅後淨利 = data.get('fundamental_features:經常稅後淨利')
權益總計 = data.get('financial_statement:股東權益總額')
營業利益成長率 = data.get('fundamental_features:營業利益成長率')
high = data.get("price:最高價")
c = BuyCondition()
自由現金流 = (df1 + df2).rolling(4).mean()
市值 = c.capitalization()
股東權益報酬率 = 稅後淨利/ 權益總計
rev = data.get('monthly_revenue:當月營收')* 1000
當季營收 = rev.rolling(4).sum()
市值營收比 = 市值 / 當季營收
rev_ma3 = rev.average(3)
rev_ma12 = rev.average(12)
#定義散戶
h1 = c.small_inv(level = 5,datatype = '持有股數')
#定義大戶
h2 = c.boss_inv(down_level = 9,  up_level= 15,datatype = '持有股數')
#大戶比例
ratio = (h2 / (h1 + h2))
s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
    (close.notna()))
rsv = (close - close.rolling(50).min()) / (close.rolling(50).max() - close.rolling(50).min())

position = (
    (自由現金流 > 0)
    & (股東權益報酬率 > 0)
    & (營業利益成長率 > -10)
    & (市值營收比 < 3 )
    & (c.max_market_capitalization(quantile=0.7))
    & (c.min_market_capitalization(quantile=0.1))
    & ((s.rank(pct=True, axis=1) > 0.3))
    & c.closing_price_below_market_classification(quantile=0.9)
    & (rev/rev.shift(12)>1)
    & (rev_ma3/rev_ma12>0.95)
    & (c.compute_candle_volatility(timeperiod=20)<7.5)
    & (vol > 150 * 1000)

    )

position = (position[position>0] * rsv).is_largest(5)
position = position.reindex(rev.index_str_to_date().index)

report = sim(
            position,
            stop_loss=0.11,
            position_limit=0.3,
            trade_at_price='close',
            upload=True,
            name='未優化抬轎小天才',
            mae_mfe_window=40,live_performance_start='2023-09-01'
            )



In [ ]:
"""
from finlab import data
from finlab import backtest
# 從 finlab data 模組中取得券商分點交易數據，強制重新下載最新數據
券商分點資料 = data.get('broker_transactions', force_download=True)

# 處理券商分點資料，將買入和賣出數據轉換為浮點數，並依日期和股票代號進行分組求和
broker_volume = (
    券商分點資料
      .reset_index()[['buy', 'sell', 'stock_id', 'date']]
      .assign(buy=lambda df: df.buy.astype(float), sell=lambda df: df.sell.astype(float))
      .groupby(['date','stock_id']).sum()
      .reset_index()
      .pivot(index='date', columns='stock_id')
)

# 刪除原始的券商分點資料，釋放記憶體
del 券商分點資料

# 計算主力買賣超的張數，即買入量減去賣出量
主力買賣超張數 = broker_volume.buy - broker_volume.sell
force = (broker_volume.buy - broker_volume.sell)/(broker_volume.buy + broker_volume.sell)
force = force[force.columns[~force.columns.str[-1].isin(['L', 'B'])]]
adj = data.get('etl:adj_close')
close = data.get('price:收盤價')
volume = data.get('price:成交股數')


pos = (
    (force.is_largest(1)).hold_until((force.rolling(5).sum() < 0), rank=force, nstocks_limit=10)

       )

report = backtest.sim(pos, resample='W', fee_ratio=1.425/1000, name="買賣超測試", upload=True)

"""

In [ ]:
"""
from finlab import backtest
import pandas as pd
from finlab import data
import talib
import numpy as np
from finlab import data
from finlab.dataframe import FinlabDataFrame
from finlab import login
import pandas as pd
import os
import scipy.stats as stats

def make_total_iterations(**params):
    return '_'.join(f'{k}_{v}' for k, v in params.items())

class BuyCondition:

    def __init__(self):
        login(os.getenv('finlab_id_token'))
        data.set_storage(data.FileStorage())
        self.close =  data.get("price:收盤價")
        self.high =  data.get("price:最高價")
        self.low =  data.get("price:最低價")
        self.open =   data.get("price:開盤價")
        self.volume =  data.get("price:成交股數")
        self.inv = data.get('inventory')


    def small_inv(self,level = 5,datatype = '持有股數'):

        small_inv = FinlabDataFrame(self.inv[self.inv.持股分級.astype(int) <= level]
                                .reset_index().groupby(['date', 'stock_id'])
                                .agg({datatype: 'sum'})
                                .reset_index()
                                .pivot(index = 'date',columns= 'stock_id')[datatype])
        return small_inv

    def boss_inv(self,down_level = 11,  up_level= 15 ,datatype = '持有股數'):

        boss_inv =  FinlabDataFrame(
            self.inv[(self.inv.持股分級.astype(int) >= down_level)
                    & (self.inv.持股分級.astype(int) <= up_level)]
                                    .reset_index()
                                    .groupby(['date', 'stock_id'])
                                    .agg({datatype: 'sum'})
                                    .reset_index()
                                    .pivot(index='date',columns= 'stock_id')[datatype])
        return boss_inv

c = BuyCondition()
close = data.get('price:收盤價')
h1 = c.small_inv(level = 5,datatype = '持有股數')
h2 = c.boss_inv(down_level = 11,  up_level= 15,datatype = '持有股數')
#大戶比例
ratio = (h2 / (h1 + h2))
s = (FinlabDataFrame(ratio.diff(6)).rank(axis=1, pct=True) *
(close.notna()))


with data.universe(market='TSE_OTC'):
    allColumns =data.get('price:收盤價').columns

收盤價 = data.get("price:收盤價")
volume = data.get('price:成交股數')

# 短期波動
std20 = 收盤價.pct_change().rolling(20).std().rank(axis=1, pct=True)[收盤價.notna()]
# 股價相對高低
rsv = (收盤價 - 收盤價.rolling(50).min()) / (收盤價.rolling(50).max() - 收盤價.rolling(50).min())
rs = (收盤價 / 收盤價.shift(10) - 1).rank(axis=1, pct=True) * 100
rank = sum([收盤價 > 收盤價.average(i) for i in range(10, 150, 20)])


# 益本比
每股盈餘 = data.get('financial_statement:每股盈餘')
益本比 = 每股盈餘 / 收盤價

rev = data.get('monthly_revenue:當月營收')* 1000
rev_ma3 = rev.average(3)
rev_ma12 = rev.average(12)
rev_ma = rev.average(2)
營利動能= (( (rev/rev.shift(12)>1.05)  & (rev_ma3/rev_ma12>0.95)
      ) | ( rev_ma == rev_ma.rolling(12, min_periods=6).max()))


股價動能 =( (收盤價 == 收盤價.rolling(200).max()).sustain(5,2)
       |(rsv.rank(axis=1, pct=True) > 0.82)
       |(std20.rank(axis=1, pct=True) < 0.92)
       |(rs > rs.quantile_row(0.45))
       |(rank.rank(axis=1, pct=True) > 0.8)
     )



def Z(df):
    df= df.apply(stats.zscore,axis=1,nan_policy='omit') # zscore, 忽略nan
    cond = (df>3) | (df<-3) # 只留下3個std以內的資料,離群值不採納
    df[cond] = np.nan
    return df

# 相關性前三名(益本比、營利動能、股價動能)
scoreTop3 = (Z(益本比) + Z(營利動能) + Z(股價動能))/3
col = scoreTop3.columns.intersection(allColumns)
scoreTop3 = scoreTop3[col].applymap(lambda z: 1+z if(z>=0) else (1-z)**-1)

sma5 = 收盤價.average(5)
drawdown = 收盤價 / 收盤價.cummax()

cond = (
    ((scoreTop3 > scoreTop3.quantile_row(95/100)))
    &((s.rank(pct=True, axis=1) > 0.3))
    &(收盤價>sma5)
    &(drawdown.rolling(500, min_periods=200).min().rank(axis=1, pct=True) > 0.5)
    )
position = scoreTop3[cond].index_str_to_date()
position = (position[position>0] * rsv).is_largest(20)
position = position.reindex(rev.index_str_to_date().index)
position.dropna(how="all",inplace=True)



report1 = backtest.sim(
            position,
            stop_loss=0.11,
            position_limit=0.3,
            trade_at_price='close',
            upload=True,
            name='超越00905',
            fee_ratio=1.425/1000*0.22,
            mae_mfe_window=40,live_performance_start='2023-09-01'
            )
"""

In [ ]:
"""
# 導入finlab中的data模塊，該模塊提供了獲取財務數據的功能
from finlab import data
import pandas as pd

# 獲取外資和投信的買賣超數據
外资买卖超股数 = data.get('institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)')
投信买卖超股数 = data.get('institutional_investors_trading_summary:投信買賣超股數')

# 定義函數，用於獲取過去幾天內買超前35名的股票代碼
def get_top_buying_stocks(data, days):
    top_stocks = []
    for i in range(1, days + 1):
        twsefb = data.iloc[-i]
        twsefb_cond = (twsefb > 0)
        twsefb_slist = twsefb_cond[twsefb_cond]
        tsefb = sorted(list(twsefb_slist.index), key=lambda v: twsefb[v], reverse=True)
        top_stocks.append(set(tsefb[:35]))
    return top_stocks

# 定義函數，用於獲取連續多日買超的股票代碼
def get_consecutive_buying_stocks(top_stocks, days):
    consecutive_buying_stocks = []
    for i in range(1, days):
        consecutive_buying = top_stocks[0].intersection(*top_stocks[1:i+1])
        filtered_stocks = [stock for stock in consecutive_buying if stock[0] != '0' and len(stock) <= 4]
        consecutive_buying_stocks.append(filtered_stocks)
    return consecutive_buying_stocks

# 獲取外資和投信過去五天買超前35名的股票代碼列表
外资_top_buying_stocks = get_top_buying_stocks(外资买卖超股数, 5)
投信_top_buying_stocks = get_top_buying_stocks(投信买卖超股数, 5)

# 獲取外資和投信連續多日買超的股票代碼列表
外资_consecutive_buying_stocks = get_consecutive_buying_stocks(外资_top_buying_stocks, 5)
投信_consecutive_buying_stocks = get_consecutive_buying_stocks(投信_top_buying_stocks, 5)

# 統計外資和投信買入最多的股票
def get_most_bought_stocks(consecutive_buying_stocks):
    stocks_dict = {}
    for stocks in consecutive_buying_stocks:
        for stock in stocks:
            if stock not in stocks_dict:
                stocks_dict[stock] = 1
            else:
                stocks_dict[stock] += 1
    most_bought_stocks = sorted(stocks_dict.items(), key=lambda x: x[1], reverse=True)
    return most_bought_stocks

# 外資連續多日買入最多的股票
外资_most_bought_stocks = get_most_bought_stocks(外资_consecutive_buying_stocks)

# 投信連續多日買入最多的股票
投信_most_bought_stocks = get_most_bought_stocks(投信_consecutive_buying_stocks)

# 格式化輸出
def format_output(most_bought_stocks):
    output = ""
    for i, (stock_code, days) in enumerate(most_bought_stocks[:5], start=1):
        output += f"{i}. 股票代碼：{stock_code}，買入天數：{days}\n"
    return output

print("外资連續多日買入最多的股票：")
print(format_output(外资_most_bought_stocks))

print("投信連續多日買入最多的股票：")
print(format_output(投信_most_bought_stocks))
"""
